# Avant le TP5 : configuration d'un client, erreurs de connexion et anatomie d'une ACL

Le TP5 fait travailler trois comptes, chacun avec son propre client Python. Cette démonstration montre, sans toucher à la sécurité du cluster :

1. comment un fichier `.properties` de client Java se traduit en configuration Python ;
2. comment lire une erreur de connexion avec la fonction `connection()` du TP ;
3. de quoi est faite une ACL, et comment on la crée, la lit et la retire ;
4. où le broker journalise ses décisions d'autorisation.

Chaque étape montre la version **terminal**, puis la version **Python**. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

Les noms et les valeurs sont définis une seule fois, ici. Dans un terminal de JupyterLab, définissez d'abord : `export PRINCIPAL=User:demo RESOURCE=demo-tp5 FILE=/opt/lab/clients/plaintext.properties`.

In [1]:
import logging
import os

from confluent_kafka import KafkaException
from confluent_kafka.admin import (AclBinding, AclBindingFilter, AclOperation, AclPermissionType, AdminClient,
                                   ResourcePatternType, ResourceType)

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}

FILE = "/opt/lab/clients/plaintext.properties"  # dans le conteneur tools
UNREACHABLE_ADDRESS = "kafka-1:9999"  # aucun listener sur ce port
PRINCIPAL = "User:demo"  # compte fictif : une ACL peut viser un compte avant sa création
RESOURCE = "demo-tp5"  # topic visé par l'ACL ; il n'a pas besoin d'exister

## 1. Du fichier Java au dictionnaire Python

Les outils Java lisent leur configuration de sécurité dans un fichier passé par `--command-config`.

**En terminal** : le fichier du listener INTERNAL, sans ses commentaires.

In [2]:
!docker exec kafka-tools grep -v '^#' $FILE

security.protocol=PLAINTEXT


**En Python**, la même configuration est un dictionnaire. On le construit à partir du fichier lui-même : `lines = !command` range la sortie dans une liste, une ligne par élément. Les noms sont souvent identiques, mais pas toujours : le TP donne un tableau de correspondance pour SASL et TLS.

In [3]:
lines = !docker exec kafka-tools grep -v '^#' $FILE
internal_config = {"bootstrap.servers": BS}
for line in lines:
    if "=" in line:
        key, value = line.split("=", 1)
        internal_config[key.strip()] = value.strip()
print(internal_config)

{'bootstrap.servers': 'kafka-1:19092,kafka-2:19092,kafka-3:19092', 'security.protocol': 'PLAINTEXT'}

## 2. Lire une erreur de connexion

En terminal, un outil Java qui ne joint pas le cluster réessaie jusqu'à expiration de son délai, puis affiche une exception de plusieurs lignes. Exemple, à lancer dans un terminal si vous voulez voir l'attente (**non exécuté ici**, une minute environ) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server kafka-1:9999 --list
```

**En Python**, librdkafka réessaie aussi sans fin et écrit une ligne de journal à chaque tentative. La fonction `connection()` du TP fait un seul essai de 10 s, range ces lignes au lieu de les afficher, et renvoie un résultat lisible : `("OK", topics)` ou `(code d'erreur, premier message)`.

In [4]:
class _LogCapture(logging.Handler):
    def __init__(self):
        super().__init__(logging.INFO)
        self.lines = []

    def emit(self, record):
        self.lines.append((record.levelno, record.getMessage().split("]: ", 1)[-1]))


def connection(config):
    log_capture = _LogCapture()
    logger = logging.getLogger("demo.connexion.%d" % id(log_capture))
    logger.propagate = False
    logger.setLevel(logging.INFO)
    logger.addHandler(log_capture)
    errors = []
    client = AdminClient({**config, "logger": logger, "error_cb": errors.append})
    try:
        return "OK", sorted(client.list_topics(timeout=10).topics)
    except KafkaException as e:
        client.poll(1)
        code = (errors[0] if errors else e.args[0]).name()
        # Le message le plus grave : une déconnexion est journalisée en INFO, un refus d'authentification en ERROR
        lines = sorted(log_capture.lines, key=lambda line: -line[0])
        return code, lines[0][1] if lines else e.args[0].str()


print(connection(internal_config))
print(connection({"bootstrap.servers": UNREACHABLE_ADDRESS}))

('OK', ['__consumer_offsets', 'audit-short', 'clickstream', 'loadgen', 'orders', 'payments', 'product-prices', 'refunds'])


('_TRANSPORT', 'kafka-1:9999/bootstrap: Connect to ipv4#172.22.0.3:9999 failed: Connection refused (after 0ms in state CONNECT)')


Le code d'erreur commence par `_` quand l'erreur est détectée par le client lui-même (`_TRANSPORT`, `_AUTHENTICATION`, `_SSL`) ; sans `_`, c'est une réponse du broker (`TOPIC_AUTHORIZATION_FAILED`).

## 3. Anatomie d'une ACL

Une ACL dit : **qui** (principal), depuis **où** (hôte), peut ou ne peut pas (permission) faire **quoi** (opération) sur **quelle ressource** (type, nom, motif). Exemple sur le compte fictif `demo`.

Création, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --add --allow-principal $PRINCIPAL \
  --operation Read --topic $RESOURCE
```

**En Python** : `create_acls()` renvoie un futur par ACL ; une ACL déjà présente n'est pas une erreur, sa création ne change rien.

In [5]:
admin = AdminClient(CONF)
acl = AclBinding(ResourceType.TOPIC, RESOURCE, ResourcePatternType.LITERAL, PRINCIPAL, "*",
                 AclOperation.READ, AclPermissionType.ALLOW)
try:
    admin.create_acls([acl])[acl].result()
    print("créé :", acl)
except KafkaException as e:
    print("Refus :", e.args[0].name(), e.args[0].str())
    raise

créé : AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)


Lecture. **En terminal**

In [6]:
!docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --list --principal $PRINCIPAL

ACLs for principal `User:demo`


Current ACLs for resource `ResourcePattern(resourceType=TOPIC, name=demo-tp5, patternType=LITERAL)`:
	(principal=User:demo, host=*, operation=READ, permissionType=ALLOW)



**En Python** : pour lire ou retirer des ACL, on passe un **filtre** : mêmes champs, mais `None` ou `ANY` veulent dire « n'importe lequel ».

In [7]:
admin = AdminClient(CONF)
acl_filter = AclBindingFilter(ResourceType.ANY, None, ResourcePatternType.ANY, PRINCIPAL, None,
                              AclOperation.ANY, AclPermissionType.ANY)
print("lues :", admin.describe_acls(acl_filter).result())

lues : [AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)]


Retrait, équivalent en terminal (**ne pas lancer en plus**) ; `--force` évite la question de confirmation :

```bash
docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --remove --allow-principal $PRINCIPAL \
  --operation Read --topic $RESOURCE --force
```

**En Python** : `delete_acls()` renvoie la liste des ACL retirées ; une liste vide n'est pas une erreur.

In [8]:
admin = AdminClient(CONF)
print("retirées :", admin.delete_acls([acl_filter])[acl_filter].result())
print("restantes :", admin.describe_acls(acl_filter).result())

retirées :

 [AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)]
restantes : []


## 4. Le journal des décisions

Chaque broker écrit ses refus d'autorisation dans `kafka-authorizer.log`, à côté de ses autres journaux. Le TP le lit avec une commande de terminal.

**En terminal** : les fichiers en cours, sans les fichiers archivés par heure, dont le nom finit par un chiffre.

In [9]:
!docker exec kafka-1 sh -c 'ls /opt/kafka/logs/ | grep -v "[0-9]$"'

controller.log
kafka-authorizer.log
kafka-request.log
kafkaServer-gc.log
log-cleaner.log
server.log
state-change.log


**En Python** : la même sortie, rangée dans une liste puis filtrée.

In [10]:
files = !docker exec kafka-1 ls /opt/kafka/logs/
print([f for f in files if not f[-1].isdigit()])

['controller.log', 'kafka-authorizer.log', 'kafka-request.log', 'kafkaServer-gc.log', 'log-cleaner.log', 'server.log', 'state-change.log']


## La méthode du TP5

Ouvrez `tps/jour2/tp5_securite/enonce.ipynb`.

- Le client d'administration reste celui du listener INTERNAL. Les comptes `admin`, `alice` et `bob` ont chacun leur configuration, par le listener SECURE.
- Les fonctions fournies `connection()`, `send()` et `read()` renvoient un code d'erreur : comparez-le à celui de l'outil Java de la version Markdown.
- Un refus est un résultat attendu dans ce TP : lisez le code, puis cherchez la ligne correspondante dans `kafka-authorizer.log`.